# PROYEK ANALISIS SENTIMEN ULASAN STEAM - COUNTER-STRIKE 2

**Nama:** Dandy Taufiqurrochman  
**NIM:** 7020230002  
**Program Studi:** Sistem Informasi  
**Perguruan Tinggi:** Universitas Galuh  
**Mata Kuliah/Program:** Belajar Fundamental Deep Learning - Dicoding

## Ringkasan
Proyek ini membangun model klasifikasi biner atas ulasan pengguna Steam untuk game Counter-Strike 2 (App ID: 730). Data dikumpulkan menggunakan Steam `GetAppReviews` API, diproses menjadi dataset pemodelan, lalu diuji menggunakan pendekatan TF-IDF, BERT, RoBERTa, dan ensemble.

## Definisi target dan batasan interpretasi
Label `positif` dan `negatif` pada proyek ini berasal dari keputusan rekomendasi pengguna Steam (`voted_up`). Karena itu, model memprediksi **rekomendasi Steam**, bukan label polaritas sentimen teks yang telah diverifikasi oleh anotator manusia.

ChatGPT digunakan sebagai alat bantu penyusunan/debugging kode dan anotasi tambahan untuk audit bahasa/kualitas data. Anotasi berbantuan AI tidak digunakan sebagai ground truth target model dan tidak diklaim sebagai anotasi manual manusia.


## 1. Dataset dan validasi

Dataset mentah berisi 10.000 ulasan hasil scraping (5.000 rekomendasi positif dan 5.000 negatif). Setelah preprocessing dan seleksi data untuk pemodelan, digunakan 3.518 ulasan: 2.462 data training, 528 validation, dan 528 testing. Pembagian data dipertahankan terpisah berdasarkan `review_id`.


In [1]:
from pathlib import Path
import uuid
from google.colab import drive

# Gunakan mount yang sudah tersedia; jika belum ada, buat mount point kosong.
mount_candidates = [
    Path('/content/gdrive_verified_e1cccd02'),
    Path('/content/drive'),
]
MOUNT_ROOT = next(
    (root for root in mount_candidates if (root / 'MyDrive').is_dir()),
    None
)

if MOUNT_ROOT is None:
    MOUNT_ROOT = Path('/content/gdrive_submission')
    if MOUNT_ROOT.exists() and any(MOUNT_ROOT.iterdir()):
        MOUNT_ROOT = Path(f'/content/gdrive_submission_{uuid.uuid4().hex[:8]}')
    MOUNT_ROOT.mkdir(parents=True, exist_ok=True)
    drive.mount(str(MOUNT_ROOT))

MYDRIVE = MOUNT_ROOT / 'MyDrive'
PROJECT = MYDRIVE / 'Proyek_Analisis_Sentimen'

required_dataset = PROJECT / 'data/final/steam_reviews_train_final.csv'
if not required_dataset.is_file():
    raise FileNotFoundError(
        'Folder proyek tidak ditemukan. Ekstrak folder Proyek_Analisis_Sentimen '
        'ke Google Drive/MyDrive terlebih dahulu.'
    )

print('Google Drive terhubung:', MYDRIVE.is_dir())
print('Folder proyek ditemukan:', PROJECT.is_dir())
print('Lokasi proyek:', PROJECT)


Google Drive terhubung: True
Folder proyek ditemukan: True
Lokasi proyek: MyDrive/Proyek_Analisis_Sentimen


In [2]:
import pandas as pd

RAW_PATH = PROJECT / 'data/raw/steam_reviews_main_10000_raw.csv'
FINAL_DIR = PROJECT / 'data/final'

raw_df = pd.read_csv(RAW_PATH, low_memory=False)
train_df = pd.read_csv(FINAL_DIR / 'steam_reviews_train_final.csv')
valid_df = pd.read_csv(FINAL_DIR / 'steam_reviews_validation_final.csv')
test_df = pd.read_csv(FINAL_DIR / 'steam_reviews_test_final.csv')
final_df = pd.read_csv(FINAL_DIR / 'steam_reviews_final.csv')

assert len(raw_df) == 10000 and raw_df['review_id'].nunique() == 10000
assert len(final_df) == 3518
assert (len(train_df), len(valid_df), len(test_df)) == (2462, 528, 528)
assert all(df[['review_id', 'text', 'label']].notna().all().all()
           for df in [train_df, valid_df, test_df])

train_ids = set(train_df['review_id'].astype(str))
valid_ids = set(valid_df['review_id'].astype(str))
test_ids = set(test_df['review_id'].astype(str))
assert not (train_ids & valid_ids or train_ids & test_ids or valid_ids & test_ids)

print('Dataset mentah:', len(raw_df), 'ulasan')
print('Dataset final:', len(final_df), 'ulasan')
print('Training:', len(train_df), '|', train_df['label'].value_counts().to_dict())
print('Validation:', len(valid_df), '|', valid_df['label'].value_counts().to_dict())
print('Testing:', len(test_df), '|', test_df['label'].value_counts().to_dict())
print('ID antar-split tumpang tindih: 0')
print('Validasi dataset: BERHASIL')


Dataset mentah: 10000 ulasan
Dataset final: 3518 ulasan
Training: 2462 | {'negatif': 1709, 'positif': 753}
Validation: 528 | {'negatif': 366, 'positif': 162}
Testing: 528 | {'negatif': 367, 'positif': 161}
ID antar-split tumpang tindih: 0
Validasi dataset: BERHASIL


## 2. Eksperimen Machine Learning - TF-IDF

Tiga skema dibandingkan pada validation set: Word TF-IDF + LinearSVC, Character TF-IDF + LinearSVC, dan Hybrid Word-Character TF-IDF + Logistic Regression.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# ============================================================
# 1. PATH DAN DATASET
# ============================================================

PROJECT = Path(
    PROJECT
)

FINAL_DIR = PROJECT / 'data/final'
MODEL_DIR = PROJECT / 'models/final_experiments'
REPORT_DIR = PROJECT / 'reports/final_experiments'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(FINAL_DIR / 'steam_reviews_train_final.csv')
valid = pd.read_csv(FINAL_DIR / 'steam_reviews_validation_final.csv')

X_train = train['text'].astype(str)
y_train = train['label'].astype(str)

X_valid = valid['text'].astype(str)
y_valid = valid['label'].astype(str)

print("Train:", len(train))
print("Validation:", len(valid))
print("Train label:", y_train.value_counts().to_dict())

# ============================================================
# 2. DEFINISI 3 SKEMA EKSPERIMEN
# ============================================================

word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    max_features=80000,
    sublinear_tf=True,
    strip_accents='unicode',
    token_pattern=r'(?u)\b\w+\b'
)

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=2,
    max_features=100000,
    sublinear_tf=True,
    strip_accents='unicode'
)

schemes = [
    {
        'name': 'S1_Word_TFIDF_LinearSVC',
        'feature': 'Word TF-IDF (unigram + bigram)',
        'algorithm': 'LinearSVC',
        'vectorizer': word_vectorizer,
        'classifier': 'svm'
    },
    {
        'name': 'S2_Char_TFIDF_LinearSVC',
        'feature': 'Character TF-IDF (3-5 gram)',
        'algorithm': 'LinearSVC',
        'vectorizer': char_vectorizer,
        'classifier': 'svm'
    },
    {
        'name': 'S3_Hybrid_TFIDF_LogisticRegression',
        'feature': 'Word + Character TF-IDF',
        'algorithm': 'Logistic Regression',
        'vectorizer': FeatureUnion([
            ('word', TfidfVectorizer(
                analyzer='word',
                ngram_range=(1, 2),
                min_df=2,
                max_features=80000,
                sublinear_tf=True,
                strip_accents='unicode'
            )),
            ('char', TfidfVectorizer(
                analyzer='char_wb',
                ngram_range=(3, 5),
                min_df=2,
                max_features=100000,
                sublinear_tf=True,
                strip_accents='unicode'
            ))
        ]),
        'classifier': 'logistic'
    }
]

C_VALUES = [0.1, 0.3, 1.0, 3.0, 10.0]
CLASS_WEIGHTS = [None, 'balanced']

all_results = []
best_models = []

# ============================================================
# 3. TRAINING DAN VALIDATION
# ============================================================

for scheme in schemes:

    print("\n" + "=" * 65)
    print("EKSPERIMEN:", scheme['name'])
    print("=" * 65)

    vectorizer = scheme['vectorizer']

    X_train_vec = vectorizer.fit_transform(X_train)
    X_valid_vec = vectorizer.transform(X_valid)

    print("Train feature shape:", X_train_vec.shape)
    print("Validation feature shape:", X_valid_vec.shape)

    scheme_results = []
    best_accuracy = -1
    best_macro_f1 = -1
    best_classifier = None
    best_config = None

    for weight in CLASS_WEIGHTS:
        for c_value in C_VALUES:

            if scheme['classifier'] == 'svm':
                classifier = LinearSVC(
                    C=c_value,
                    class_weight=weight,
                    max_iter=20000,
                    random_state=42
                )
            else:
                classifier = LogisticRegression(
                    C=c_value,
                    class_weight=weight,
                    max_iter=3000,
                    solver='liblinear',
                    random_state=42
                )

            classifier.fit(X_train_vec, y_train)
            predictions = classifier.predict(X_valid_vec)

            accuracy = accuracy_score(y_valid, predictions)
            balanced = balanced_accuracy_score(
                y_valid, predictions
            )
            macro_f1 = f1_score(
                y_valid, predictions, average='macro'
            )
            weighted_f1 = f1_score(
                y_valid, predictions, average='weighted'
            )

            result = {
                'scheme': scheme['name'],
                'algorithm': scheme['algorithm'],
                'feature': scheme['feature'],
                'C': c_value,
                'class_weight': str(weight),
                'validation_accuracy': accuracy,
                'balanced_accuracy': balanced,
                'macro_f1': macro_f1,
                'weighted_f1': weighted_f1
            }

            scheme_results.append(result)
            all_results.append(result)

            print(
                f"C={c_value:<5} "
                f"weight={str(weight):<9} "
                f"accuracy={accuracy:.4f} "
                f"macro_f1={macro_f1:.4f}"
            )

            if (
                accuracy > best_accuracy
                or (
                    accuracy == best_accuracy
                    and macro_f1 > best_macro_f1
                )
            ):
                best_accuracy = accuracy
                best_macro_f1 = macro_f1
                best_classifier = classifier
                best_config = result.copy()

    # Simpan model terbaik untuk setiap skema
    model_path = MODEL_DIR / f"{scheme['name']}.joblib"

    joblib.dump({
        'vectorizer': vectorizer,
        'classifier': best_classifier,
        'config': best_config,
        'classes': list(best_classifier.classes_)
    }, model_path)

    best_models.append(best_config)

    print("\nMODEL TERBAIK:", scheme['name'])
    print("Validation accuracy:", round(best_accuracy, 4))
    print("Macro F1:", round(best_macro_f1, 4))
    print("Model tersimpan:", model_path)

# ============================================================
# 4. SIMPAN HASIL PERBANDINGAN
# ============================================================

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    ['validation_accuracy', 'macro_f1'],
    ascending=False
)

best_df = pd.DataFrame(best_models).sort_values(
    ['validation_accuracy', 'macro_f1'],
    ascending=False
)

results_df.to_csv(
    REPORT_DIR / 'all_experiment_results.csv',
    index=False
)

best_df.to_csv(
    REPORT_DIR / 'best_three_schemes.csv',
    index=False
)

print("\n" + "=" * 65)
print("PERBANDINGAN 3 SKEMA TERBAIK")
print("=" * 65)

display(
    best_df[
        [
            'scheme',
            'algorithm',
            'feature',
            'C',
            'class_weight',
            'validation_accuracy',
            'balanced_accuracy',
            'macro_f1',
            'weighted_f1'
        ]
    ].reset_index(drop=True)
)

print("\nSeluruh eksperimen selesai.")
print("Lokasi model:", MODEL_DIR)
print("Lokasi laporan:", REPORT_DIR)

Train: 2462
Validation: 528
Train label: {'negatif': 1709, 'positif': 753}

EKSPERIMEN: S1_Word_TFIDF_LinearSVC
Train feature shape: (2462, 13353)
Validation feature shape: (528, 13353)
C=0.1   weight=None      accuracy=0.8087 macro_f1=0.7269
C=0.3   weight=None      accuracy=0.8239 macro_f1=0.7703
C=1.0   weight=None      accuracy=0.8182 macro_f1=0.7728
C=3.0   weight=None      accuracy=0.8163 macro_f1=0.7728
C=10.0  weight=None      accuracy=0.8201 macro_f1=0.7802
C=0.1   weight=balanced  accuracy=0.8352 macro_f1=0.8079
C=0.3   weight=balanced  accuracy=0.8371 macro_f1=0.8079
C=1.0   weight=balanced  accuracy=0.8409 macro_f1=0.8103
C=3.0   weight=balanced  accuracy=0.8333 macro_f1=0.7998
C=10.0  weight=balanced  accuracy=0.8239 macro_f1=0.7896

MODEL TERBAIK: S1_Word_TFIDF_LinearSVC
Validation accuracy: 0.8409
Macro F1: 0.8103
Model tersimpan: /content/gdrive_actual/MyDrive/Proyek_Analisis_Sentimen/models/final_experiments/S1_Word_TFIDF_LinearSVC.joblib

EKSPERIMEN: S2_Char_TFIDF_Lin

,scheme,algorithm,feature,C,class_weight,validation_accuracy,balanced_accuracy,macro_f1,weighted_f1
0,S3_Hybrid_TFIDF_LogisticRegression,Logistic Regression,Word + Character TF-IDF,3.0,balanced,0.848485,0.830500,0.824838,0.849704
1,S1_Word_TFIDF_LinearSVC,LinearSVC,Word TF-IDF (unigram + bigram),1.0,balanced,0.840909,0.806112,0.810332,0.839755
2,S2_Char_TFIDF_LinearSVC,LinearSVC,Character TF-IDF (3-5 gram),0.3,balanced,0.831439,0.819925,0.808427,0.834080



Seluruh eksperimen selesai.
Lokasi model: /content/gdrive_actual/MyDrive/Proyek_Analisis_Sentimen/models/final_experiments
Lokasi laporan: /content/gdrive_actual/MyDrive/Proyek_Analisis_Sentimen/reports/final_experiments


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import json

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

PROJECT = Path(
    PROJECT
)

MODEL_DIR = PROJECT / 'models/final_experiments'
REPORT_DIR = PROJECT / 'reports/final_experiments'

# Load data validation
valid = pd.read_csv(
    PROJECT / 'data/final/steam_reviews_validation_final.csv'
)

X_valid = valid['text'].astype(str)
y_valid = valid['label'].astype(str)

# Load model S3 terbaik
model_path = MODEL_DIR / 'S3_Hybrid_TFIDF_LogisticRegression.joblib'
artifact = joblib.load(model_path)

vectorizer = artifact['vectorizer']
classifier = artifact['classifier']

# Transform dan prediksi probabilitas
X_valid_vec = vectorizer.transform(X_valid)
probabilities = classifier.predict_proba(X_valid_vec)

positive_index = list(classifier.classes_).index('positif')
positive_prob = probabilities[:, positive_index]

# Cari threshold terbaik
thresholds = np.arange(0.05, 0.951, 0.005)

results = []

for threshold in thresholds:
    predictions = np.where(
        positive_prob >= threshold,
        'positif',
        'negatif'
    )

    results.append({
        'threshold': round(float(threshold), 3),
        'accuracy': accuracy_score(y_valid, predictions),
        'balanced_accuracy': balanced_accuracy_score(
            y_valid, predictions
        ),
        'macro_f1': f1_score(
            y_valid, predictions, average='macro'
        ),
        'weighted_f1': f1_score(
            y_valid, predictions, average='weighted'
        ),
        'precision_macro': precision_score(
            y_valid, predictions, average='macro', zero_division=0
        ),
        'recall_macro': recall_score(
            y_valid, predictions, average='macro', zero_division=0
        )
    })

results_df = pd.DataFrame(results)

# Pilih accuracy tertinggi, lalu macro F1 sebagai tie-breaker
results_df = results_df.sort_values(
    ['accuracy', 'macro_f1'],
    ascending=False
).reset_index(drop=True)

best = results_df.iloc[0]

# Prediksi dengan threshold terbaik
best_threshold = float(best['threshold'])

best_predictions = np.where(
    positive_prob >= best_threshold,
    'positif',
    'negatif'
)

# Simpan hasil
results_df.to_csv(
    REPORT_DIR / 'S3_threshold_optimization_validation.csv',
    index=False
)

with open(
    REPORT_DIR / 'S3_best_threshold.json', 'w'
) as f:
    json.dump({
        'model': 'Hybrid TF-IDF + Logistic Regression',
        'selected_threshold': best_threshold,
        'validation_accuracy': float(best['accuracy']),
        'validation_macro_f1': float(best['macro_f1'])
    }, f, indent=4)

print("=" * 60)
print("HASIL OPTIMASI THRESHOLD S3")
print("=" * 60)

print("Threshold terbaik:", best_threshold)
print("Validation accuracy:", round(best['accuracy'] * 100, 2), "%")
print("Balanced accuracy:", round(best['balanced_accuracy'], 4))
print("Macro F1:", round(best['macro_f1'], 4))
print("Weighted F1:", round(best['weighted_f1'], 4))

print("\nClassification Report:")
print(classification_report(
    y_valid,
    best_predictions,
    labels=['negatif', 'positif'],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    y_valid,
    best_predictions,
    labels=['negatif', 'positif']
))

print("\n10 threshold terbaik:")
display(results_df.head(10))

print("\nHasil tersimpan di:", REPORT_DIR)

HASIL OPTIMASI THRESHOLD S3
Threshold terbaik: 0.505
Validation accuracy: 85.04 %
Balanced accuracy: 0.8301
Macro F1: 0.8262
Weighted F1: 0.8512

Classification Report:
              precision    recall  f1-score   support

     negatif       0.90      0.88      0.89       366
     positif       0.75      0.78      0.76       162

    accuracy                           0.85       528
   macro avg       0.82      0.83      0.83       528
weighted avg       0.85      0.85      0.85       528

Confusion Matrix:
[[323  43]
 [ 36 126]]

10 threshold terbaik:


,threshold,accuracy,balanced_accuracy,macro_f1,weighted_f1,precision_macro,recall_macro
0,0.505,0.850379,0.830146,0.826182,0.851239,0.822642,0.830146
1,0.510,0.850379,0.830146,0.826182,0.851239,0.822642,0.830146
2,0.515,0.850379,0.828425,0.825606,0.851001,0.823005,0.828425
3,0.500,0.848485,0.830500,0.824838,0.849704,0.820029,0.830500
4,0.525,0.846591,0.820532,0.819981,0.846722,0.819439,0.820532
5,0.530,0.846591,0.818812,0.819361,0.846458,0.819918,0.818812
6,0.495,0.844697,0.827768,0.821032,0.846176,0.815508,0.827768
7,0.520,0.844697,0.819166,0.818069,0.844961,0.817006,0.819166
8,0.535,0.844697,0.815726,0.816812,0.844426,0.817935,0.815726
9,0.540,0.840909,0.809553,0.811685,0.840347,0.813958,0.809553



Hasil tersimpan di: /content/gdrive_actual/MyDrive/Proyek_Analisis_Sentimen/reports/final_experiments


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import json

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# =====================================================
# 1. PATH
# =====================================================

PROJECT = Path(
    PROJECT
)

MODEL_DIR = PROJECT / 'models/final_experiments'
REPORT_DIR = PROJECT / 'reports/final_experiments'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# 2. LOAD MODEL DAN TEST DATA
# =====================================================

MODEL_PATH = (
    MODEL_DIR / 'S3_Hybrid_TFIDF_LogisticRegression.joblib'
)

TEST_PATH = (
    PROJECT / 'data/final/steam_reviews_test_final.csv'
)

artifact = joblib.load(MODEL_PATH)
test = pd.read_csv(TEST_PATH)

vectorizer = artifact['vectorizer']
classifier = artifact['classifier']

X_test = test['text'].astype(str)
y_test = test['label'].astype(str)

THRESHOLD = 0.505

assert len(test) == 528
assert test['review_id'].nunique() == 528

# =====================================================
# 3. PREDIKSI TEST
# =====================================================

X_test_vec = vectorizer.transform(X_test)

probabilities = classifier.predict_proba(X_test_vec)

positive_index = list(classifier.classes_).index('positif')
positive_prob = probabilities[:, positive_index]

predictions = np.where(
    positive_prob >= THRESHOLD,
    'positif',
    'negatif'
)

# =====================================================
# 4. EVALUASI
# =====================================================

accuracy = accuracy_score(y_test, predictions)
balanced = balanced_accuracy_score(y_test, predictions)
macro_f1 = f1_score(y_test, predictions, average='macro')
weighted_f1 = f1_score(y_test, predictions, average='weighted')

precision_macro = precision_score(
    y_test, predictions, average='macro', zero_division=0
)

recall_macro = recall_score(
    y_test, predictions, average='macro', zero_division=0
)

report = classification_report(
    y_test,
    predictions,
    labels=['negatif', 'positif'],
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    y_test,
    predictions,
    labels=['negatif', 'positif']
)

# =====================================================
# 5. SIMPAN PREDIKSI
# =====================================================

prediction_df = test[
    ['review_id', 'review', 'text', 'label', 'split']
].copy()

prediction_df['predicted_label'] = predictions
prediction_df['positive_probability'] = positive_prob
prediction_df['threshold'] = THRESHOLD
prediction_df['correct'] = (
    prediction_df['label'] == prediction_df['predicted_label']
)

prediction_df.to_csv(
    REPORT_DIR / 'final_test_predictions.csv',
    index=False
)

# =====================================================
# 6. SIMPAN METRIK
# =====================================================

metrics = {
    'model': 'Hybrid TF-IDF + Logistic Regression',
    'feature_extraction': 'Word TF-IDF + Character TF-IDF',
    'threshold': THRESHOLD,
    'test_samples': len(test),
    'accuracy': float(accuracy),
    'balanced_accuracy': float(balanced),
    'macro_f1': float(macro_f1),
    'weighted_f1': float(weighted_f1),
    'precision_macro': float(precision_macro),
    'recall_macro': float(recall_macro),
    'correct_predictions': int((predictions == y_test).sum()),
    'incorrect_predictions': int((predictions != y_test).sum()),
    'confusion_matrix': cm.tolist()
}

with open(REPORT_DIR / 'final_test_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

pd.DataFrame(report).transpose().to_csv(
    REPORT_DIR / 'final_test_classification_report.csv'
)

pd.DataFrame(
    cm,
    index=['actual_negatif', 'actual_positif'],
    columns=['predicted_negatif', 'predicted_positif']
).to_csv(
    REPORT_DIR / 'final_test_confusion_matrix.csv'
)

# =====================================================
# 7. SIMPAN MODEL FINAL DAN THRESHOLD
# =====================================================

final_artifact = {
    'vectorizer': vectorizer,
    'classifier': classifier,
    'threshold': THRESHOLD,
    'classes': list(classifier.classes_),
    'model_name': 'S3 Hybrid TF-IDF Logistic Regression',
    'target': 'steam_recommendation_label',
    'validation_accuracy': 0.850379
}

joblib.dump(
    final_artifact,
    MODEL_DIR / 'cs2_sentiment_final_model.joblib'
)

# =====================================================
# 8. TAMPILKAN HASIL
# =====================================================

print('=' * 60)
print('FINAL TEST EVALUATION')
print('=' * 60)

print('Test samples:', len(test))
print('Threshold:', THRESHOLD)
print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Balanced Accuracy: {balanced * 100:.2f}%')
print(f'Macro F1: {macro_f1:.4f}')
print(f'Weighted F1: {weighted_f1:.4f}')

print('\nClassification Report:')
display(pd.DataFrame(report).transpose())

print('\nConfusion Matrix:')
display(pd.DataFrame(
    cm,
    index=['Actual negatif', 'Actual positif'],
    columns=['Predicted negatif', 'Predicted positif']
))

print('\nCorrect:', int((predictions == y_test).sum()))
print('Incorrect:', int((predictions != y_test).sum()))

print('\nModel final dan seluruh laporan berhasil disimpan.')
print('Lokasi:', REPORT_DIR)

FINAL TEST EVALUATION
Test samples: 528
Threshold: 0.505
Accuracy: 81.06%
Balanced Accuracy: 79.05%
Macro F1: 0.7824
Weighted F1: 0.8130

Classification Report:


,precision,recall,f1-score,support
negatif,0.880342,0.841962,0.860724,367.000000
positif,0.672316,0.739130,0.704142,161.000000
accuracy,0.810606,0.810606,0.810606,0.810606
macro avg,0.776329,0.790546,0.782433,528.000000
weighted avg,0.816910,0.810606,0.812979,528.000000



Confusion Matrix:


,Predicted negatif,Predicted positif
Actual negatif,309,58
Actual positif,42,119



Correct: 428
Incorrect: 100

Model final dan seluruh laporan berhasil disimpan.
Lokasi: /content/gdrive_actual/MyDrive/Proyek_Analisis_Sentimen/reports/final_experiments


### Catatan hasil TF-IDF
Skema S3 memperoleh akurasi validation 84,85%. Setelah threshold dioptimalkan pada validation (0,505), akurasi pada test untuk eksperimen S3 tersebut adalah 81,06%. Karena itu, model TF-IDF tidak dipilih sebagai model akhir. Hasil model akhir dilaporkan terpisah pada bagian ensemble.


## 3. Persiapan lingkungan Deep Learning

Jalankan dengan runtime GPU (misalnya T4 di Google Colab) untuk mengulang fine-tuning transformer. Output training yang tersimpan di notebook berasal dari proses eksperimen yang telah dijalankan.

In [6]:
%pip install -q --upgrade transformers accelerate safetensors

import torch
import transformers
import accelerate

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL_NAME = "google-bert/bert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load pretrained BERT untuk klasifikasi 2 kelas
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "negatif",
        1: "positif"
    },
    label2id={
        "negatif": 0,
        "positif": 1
    }
)

print("\nBERT berhasil dimuat.")
print("Model:", MODEL_NAME)
print("Jumlah parameter:", sum(p.numel() for p in model.parameters()))
print("Tokenizer vocabulary:", tokenizer.vocab_size)

del model
torch.cuda.empty_cache()

Transformers: 5.22.2
Accelerate: 1.15.0
PyTorch: 2.14.1+cu130
CUDA: True
GPU: Tesla T4


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



BERT berhasil dimuat.
Model: google-bert/bert-base-uncased
Jumlah parameter: 109483778
Tokenizer vocabulary: 30522


In [7]:
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Memori GPU:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("GPU belum aktif. Aktifkan melalui Runtime > Change runtime type > T4 GPU.")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
CUDA tersedia: True
GPU: Tesla T4
Memori GPU: 14.56 GB


## 4. Training dan evaluasi BERT baseline

Model BERT base uncased dilatih menggunakan data training dan dipilih berdasarkan performa validation.

In [8]:

import json
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

# =====================================================
# 1. KONFIGURASI
# =====================================================

SEED = 42
MODEL_NAME = "google-bert/bert-base-uncased"

BATCH_SIZE = 16
MAX_LENGTH = 192
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

LABEL2ID = {"negatif": 0, "positif": 1}
ID2LABEL = {0: "negatif", 1: "positif"}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "GPU CUDA tidak tersedia."

device = torch.device("cuda")

# Gunakan mount point yang sudah berhasil diverifikasi
PROJECT = Path(
    PROJECT
)

FINAL_DIR = PROJECT / "data/final"
MODEL_DIR = PROJECT / "models/bert_base_uncased_cs2"
REPORT_DIR = PROJECT / "reports/bert_base_uncased_cs2"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# 2. LOAD DATA
# =====================================================

train_df = pd.read_csv(
    FINAL_DIR / "steam_reviews_train_final.csv"
)

valid_df = pd.read_csv(
    FINAL_DIR / "steam_reviews_validation_final.csv"
)

assert len(train_df) == 2462
assert len(valid_df) == 528
assert train_df["text"].notna().all()
assert valid_df["text"].notna().all()

assert set(train_df["label"]).issubset(LABEL2ID)
assert set(valid_df["label"]).issubset(LABEL2ID)

print("Train:", len(train_df))
print("Validation:", len(valid_df))
print("GPU:", torch.cuda.get_device_name(0))

# =====================================================
# 3. TOKENIZER
# =====================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# =====================================================
# 4. DATASET
# =====================================================

class ReviewDataset(Dataset):

    def __init__(self, dataframe):
        texts = dataframe["text"].astype(str).tolist()

        self.encodings = tokenizer(
            texts,
            max_length=MAX_LENGTH,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        self.labels = torch.tensor(
            [LABEL2ID[x] for x in dataframe["label"]],
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {
            key: value[index]
            for key, value in self.encodings.items()
        }
        item["labels"] = self.labels[index]
        return item


train_dataset = ReviewDataset(train_df)
valid_dataset = ReviewDataset(valid_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# =====================================================
# 5. LOAD BERT
# =====================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(device)

# =====================================================
# 6. OPTIMIZER DAN SCHEDULER
# =====================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)

    return max(
        0.0,
        (total_steps - step) /
        max(1, total_steps - warmup_steps)
    )

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda
)

scaler = torch.amp.GradScaler("cuda")

# =====================================================
# 7. EVALUATION FUNCTION
# =====================================================

def evaluate_model(model, loader):

    model.eval()

    total_loss = 0
    actual = []
    probabilities = []

    with torch.no_grad():

        for batch in loader:

            batch = {
                k: v.to(device, non_blocking=True)
                for k, v in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**batch)

            total_loss += (
                outputs.loss.item() * len(batch["labels"])
            )

            probs = torch.softmax(
                outputs.logits.float(), dim=1
            )

            probabilities.extend(
                probs[:, LABEL2ID["positif"]].cpu().numpy()
            )

            actual.extend(
                batch["labels"].cpu().numpy()
            )

    probabilities = np.array(probabilities)
    actual = np.array(actual)

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(actual, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            actual, predictions
        ),
        "macro_f1": f1_score(
            actual, predictions, average="macro"
        ),
        "weighted_f1": f1_score(
            actual, predictions, average="weighted"
        ),
        "probabilities": probabilities,
        "actual": actual
    }

# =====================================================
# 8. TRAINING
# =====================================================

history = []

best_accuracy = -1
best_macro_f1 = -1
best_epoch = None
best_probabilities = None

for epoch in range(EPOCHS):

    print("\n" + "=" * 60)
    print(f"EPOCH {epoch + 1}/{EPOCHS}")
    print("=" * 60)

    model.train()
    total_train_loss = 0

    for step, batch in enumerate(train_loader):

        batch = {
            k: v.to(device, non_blocking=True)
            for k, v in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), 1.0
        )

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_train_loss += loss.item()

        if (step + 1) % 50 == 0:
            print(
                f"Batch {step + 1}/{len(train_loader)} "
                f"| Loss: {loss.item():.4f}"
            )

    train_loss = total_train_loss / len(train_loader)

    # Validation
    metrics = evaluate_model(model, valid_loader)

    record = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_loss": metrics["loss"],
        "validation_accuracy": metrics["accuracy"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"]
    }

    history.append(record)

    print(f"\nTrain Loss: {train_loss:.4f}")
    print(f"Validation Loss: {metrics['loss']:.4f}")
    print(f"Validation Accuracy: {metrics['accuracy']*100:.2f}%")
    print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}")
    print(f"Weighted F1: {metrics['weighted_f1']:.4f}")

    # Checkpoint terbaik berdasarkan validation accuracy
    if (
        metrics["accuracy"] > best_accuracy
        or (
            metrics["accuracy"] == best_accuracy
            and metrics["macro_f1"] > best_macro_f1
        )
    ):

        best_accuracy = metrics["accuracy"]
        best_macro_f1 = metrics["macro_f1"]
        best_epoch = epoch + 1
        best_probabilities = metrics["probabilities"].copy()

        model.save_pretrained(str(MODEL_DIR))
        tokenizer.save_pretrained(str(MODEL_DIR))

        print("CHECKPOINT TERBAIK TERSIMPAN.")

    pd.DataFrame(history).to_csv(
        REPORT_DIR / "bert_training_history.csv",
        index=False
    )

# =====================================================
# 9. VALIDATION MODEL TERBAIK
# =====================================================

best_predictions = np.where(
    best_probabilities >= 0.5,
    "positif",
    "negatif"
)

actual_labels = valid_df["label"].astype(str).values

validation_output = valid_df[
    ["review_id", "text", "label"]
].copy()

validation_output["predicted_label"] = best_predictions
validation_output["positive_probability"] = best_probabilities

validation_output.to_csv(
    REPORT_DIR / "bert_validation_predictions.csv",
    index=False
)

summary = {
    "model": MODEL_NAME,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "learning_rate": LEARNING_RATE,
    "best_epoch": best_epoch,
    "best_validation_accuracy": float(best_accuracy),
    "best_validation_macro_f1": float(best_macro_f1),
    "gpu": torch.cuda.get_device_name(0)
}

with open(
    REPORT_DIR / "bert_training_summary.json", "w"
) as f:
    json.dump(summary, f, indent=4)

# =====================================================
# 10. HASIL
# =====================================================

print("\n" + "=" * 60)
print("BERT FINE-TUNING SELESAI")
print("=" * 60)

print("Best Epoch:", best_epoch)
print(f"Best Validation Accuracy: {best_accuracy*100:.2f}%")
print(f"Best Validation Macro F1: {best_macro_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    actual_labels,
    best_predictions,
    labels=["negatif", "positif"],
    zero_division=0
))

print("\nModel:", MODEL_DIR)
print("Laporan:", REPORT_DIR)

torch.cuda.empty_cache()


Train: 2462
Validation: 528
GPU: Tesla T4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



EPOCH 1/3
Batch 50/154 | Loss: 0.4714
Batch 100/154 | Loss: 0.7396
Batch 150/154 | Loss: 0.1465

Train Loss: 0.4946
Validation Loss: 0.3517
Validation Accuracy: 86.17%
Balanced Accuracy: 0.8246
Macro F1: 0.8330
Weighted F1: 0.8598


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CHECKPOINT TERBAIK TERSIMPAN.

EPOCH 2/3
Batch 50/154 | Loss: 0.3699
Batch 100/154 | Loss: 0.1037
Batch 150/154 | Loss: 0.0952

Train Loss: 0.2842
Validation Loss: 0.3958
Validation Accuracy: 86.36%
Balanced Accuracy: 0.8053
Macro F1: 0.8266
Weighted F1: 0.8576


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CHECKPOINT TERBAIK TERSIMPAN.

EPOCH 3/3
Batch 50/154 | Loss: 0.2696
Batch 100/154 | Loss: 0.1390
Batch 150/154 | Loss: 0.0780

Train Loss: 0.1786
Validation Loss: 0.3877
Validation Accuracy: 87.12%
Balanced Accuracy: 0.8538
Macro F1: 0.8501
Weighted F1: 0.8719


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CHECKPOINT TERBAIK TERSIMPAN.

BERT FINE-TUNING SELESAI
Best Epoch: 3
Best Validation Accuracy: 87.12%
Best Validation Macro F1: 0.8501

Classification Report:
              precision    recall  f1-score   support

     negatif       0.91      0.90      0.91       366
     positif       0.78      0.81      0.79       162

    accuracy                           0.87       528
   macro avg       0.85      0.85      0.85       528
weighted avg       0.87      0.87      0.87       528


Model: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/models/bert_base_uncased_cs2
Laporan: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/reports/bert_base_uncased_cs2


In [9]:
import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# =====================================================
# 1. PATH DAN KONFIGURASI
# =====================================================

PROJECT = Path(
    PROJECT
)

MODEL_DIR = PROJECT / "models/bert_base_uncased_cs2"
REPORT_DIR = PROJECT / "reports/bert_base_uncased_cs2"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

TEST_PATH = PROJECT / "data/final/steam_reviews_test_final.csv"

MAX_LENGTH = 192
BATCH_SIZE = 16

LABEL2ID = {"negatif": 0, "positif": 1}
ID2LABEL = {0: "negatif", 1: "positif"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================
# 2. LOAD TEST DATASET
# =====================================================

test_df = pd.read_csv(TEST_PATH)

assert len(test_df) == 528
assert test_df["review_id"].nunique() == 528
assert test_df["text"].notna().all()
assert test_df["label"].isin(LABEL2ID).all()

print("Test samples:", len(test_df))
print("Device:", device)

# =====================================================
# 3. LOAD MODEL DAN TOKENIZER TERBAIK
# =====================================================

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))

model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR)
).to(device)

model.eval()

# =====================================================
# 4. TOKENIZATION
# =====================================================

class TestDataset(Dataset):

    def __init__(self, dataframe):
        self.encodings = tokenizer(
            dataframe["text"].astype(str).tolist(),
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        self.labels = torch.tensor(
            [LABEL2ID[x] for x in dataframe["label"]],
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {
            key: value[index]
            for key, value in self.encodings.items()
        }
        item["labels"] = self.labels[index]
        return item


test_dataset = TestDataset(test_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# =====================================================
# 5. PREDIKSI TEST
# =====================================================

actual = []
predictions = []
positive_probabilities = []

with torch.no_grad():

    for batch in test_loader:

        labels = batch.pop("labels").numpy()

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=torch.cuda.is_available()
        ):
            outputs = model(**batch)

        probabilities = torch.softmax(
            outputs.logits.float(),
            dim=1
        )

        predicted_ids = torch.argmax(
            probabilities, dim=1
        ).cpu().numpy()

        actual.extend(labels)
        predictions.extend(predicted_ids)
        positive_probabilities.extend(
            probabilities[:, LABEL2ID["positif"]].cpu().numpy()
        )

actual = np.array(actual)
predictions = np.array(predictions)
positive_probabilities = np.array(positive_probabilities)

y_true = [ID2LABEL[x] for x in actual]
y_pred = [ID2LABEL[x] for x in predictions]

# =====================================================
# 6. METRICS
# =====================================================

accuracy = accuracy_score(y_true, y_pred)
balanced = balanced_accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true, y_pred, average="macro", zero_division=0
)

recall = recall_score(
    y_true, y_pred, average="macro", zero_division=0
)

macro_f1 = f1_score(
    y_true, y_pred, average="macro", zero_division=0
)

weighted_f1 = f1_score(
    y_true, y_pred, average="weighted", zero_division=0
)

report = classification_report(
    y_true,
    y_pred,
    labels=["negatif", "positif"],
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["negatif", "positif"]
)

# =====================================================
# 7. SIMPAN PREDIKSI
# =====================================================

prediction_df = test_df[
    ["review_id", "review", "text", "label", "split"]
].copy()

prediction_df["predicted_label"] = y_pred
prediction_df["positive_probability"] = positive_probabilities
prediction_df["correct"] = (
    prediction_df["label"] == prediction_df["predicted_label"]
)

prediction_df.to_csv(
    REPORT_DIR / "bert_final_test_predictions.csv",
    index=False
)

# =====================================================
# 8. SIMPAN METRICS
# =====================================================

metrics = {
    "model": "BERT Base Uncased",
    "pretrained_model": "google-bert/bert-base-uncased",
    "target": "steam_recommendation_label",
    "best_epoch": 3,
    "max_length": MAX_LENGTH,
    "test_samples": len(test_df),
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced),
    "precision_macro": float(precision),
    "recall_macro": float(recall),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "correct": int(np.sum(predictions == actual)),
    "incorrect": int(np.sum(predictions != actual)),
    "confusion_matrix": cm.tolist()
}

with open(
    REPORT_DIR / "bert_final_test_metrics.json", "w"
) as f:
    json.dump(metrics, f, indent=4)

pd.DataFrame(report).transpose().to_csv(
    REPORT_DIR / "bert_final_test_classification_report.csv"
)

pd.DataFrame(
    cm,
    index=["actual_negatif", "actual_positif"],
    columns=["predicted_negatif", "predicted_positif"]
).to_csv(
    REPORT_DIR / "bert_final_test_confusion_matrix.csv"
)

# =====================================================
# 9. HASIL
# =====================================================

print("\n" + "=" * 60)
print("BERT FINAL TEST EVALUATION")
print("=" * 60)

print("Test samples:", len(test_df))
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Balanced Accuracy: {balanced * 100:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=["negatif", "positif"],
    zero_division=0
))

print("Confusion Matrix:")
print(cm)

print("\nCorrect:", int(np.sum(predictions == actual)))
print("Incorrect:", int(np.sum(predictions != actual)))

print("\nModel:", MODEL_DIR)
print("Reports:", REPORT_DIR)

print("\nEvaluasi selesai.")

Test samples: 528
Device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


BERT FINAL TEST EVALUATION
Test samples: 528
Accuracy: 84.66%
Balanced Accuracy: 83.04%
Macro F1: 0.8229
Weighted F1: 0.8482

Classification Report:
              precision    recall  f1-score   support

     negatif       0.90      0.87      0.89       367
     positif       0.73      0.79      0.76       161

    accuracy                           0.85       528
   macro avg       0.82      0.83      0.82       528
weighted avg       0.85      0.85      0.85       528

Confusion Matrix:
[[320  47]
 [ 34 127]]

Correct: 447
Incorrect: 81

Model: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/models/bert_base_uncased_cs2
Reports: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/reports/bert_base_uncased_cs2

Evaluasi selesai.


In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

PROJECT = Path(
    PROJECT
)

REPORT_DIR = PROJECT / 'reports/bert_base_uncased_cs2'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load hasil prediksi validation dan test dari checkpoint terbaik
valid = pd.read_csv(
    REPORT_DIR / 'bert_validation_predictions.csv'
)

test = pd.read_csv(
    REPORT_DIR / 'bert_final_test_predictions.csv'
)

# Validasi
assert len(valid) == 528
assert len(test) == 528
assert valid['positive_probability'].notna().all()
assert test['positive_probability'].notna().all()

y_valid = valid['label'].astype(str).values
y_test = test['label'].astype(str).values

# =====================================================
# 1. OPTIMASI THRESHOLD BERDASARKAN VALIDATION
# =====================================================

thresholds = np.arange(0.30, 0.701, 0.001)
results = []

for threshold in thresholds:

    pred = np.where(
        valid['positive_probability'].values >= threshold,
        'positif',
        'negatif'
    )

    results.append({
        'threshold': round(float(threshold), 3),
        'accuracy': accuracy_score(y_valid, pred),
        'balanced_accuracy': balanced_accuracy_score(y_valid, pred),
        'macro_f1': f1_score(y_valid, pred, average='macro'),
        'weighted_f1': f1_score(y_valid, pred, average='weighted')
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    ['accuracy', 'macro_f1'],
    ascending=False
).reset_index(drop=True)

best = results_df.iloc[0]
best_threshold = float(best['threshold'])

# Simpan hasil pencarian threshold
results_df.to_csv(
    REPORT_DIR / 'bert_threshold_search_validation.csv',
    index=False
)

# =====================================================
# 2. TERAPKAN THRESHOLD PADA TEST
# =====================================================

test_predictions = np.where(
    test['positive_probability'].values >= best_threshold,
    'positif',
    'negatif'
)

accuracy = accuracy_score(y_test, test_predictions)
balanced = balanced_accuracy_score(y_test, test_predictions)
macro_f1 = f1_score(y_test, test_predictions, average='macro')
weighted_f1 = f1_score(y_test, test_predictions, average='weighted')

cm = confusion_matrix(
    y_test,
    test_predictions,
    labels=['negatif', 'positif']
)

# =====================================================
# 3. SIMPAN HASIL FINAL
# =====================================================

test['predicted_label'] = test_predictions
test['threshold'] = best_threshold
test['correct'] = test['label'] == test_predictions

test.to_csv(
    REPORT_DIR / 'bert_final_test_predictions_optimized.csv',
    index=False
)

metrics = {
    'model': 'BERT Base Uncased',
    'threshold_source': 'validation set',
    'threshold': best_threshold,
    'validation_accuracy': float(best['accuracy']),
    'test_accuracy': float(accuracy),
    'test_balanced_accuracy': float(balanced),
    'test_macro_f1': float(macro_f1),
    'test_weighted_f1': float(weighted_f1),
    'test_samples': len(test),
    'correct': int((test_predictions == y_test).sum()),
    'incorrect': int((test_predictions != y_test).sum()),
    'confusion_matrix': cm.tolist()
}

with open(
    REPORT_DIR / 'bert_final_optimized_metrics.json', 'w'
) as f:
    json.dump(metrics, f, indent=4)

print('=' * 60)
print('OPTIMASI THRESHOLD BERT')
print('=' * 60)

print('Threshold terbaik:', best_threshold)
print('Validation accuracy:', f"{best['accuracy']*100:.2f}%")

print('\nFINAL TEST DENGAN THRESHOLD TERPILIH')
print('Test accuracy:', f"{accuracy*100:.2f}%")
print('Balanced accuracy:', f"{balanced*100:.2f}%")
print('Macro F1:', round(macro_f1, 4))
print('Weighted F1:', round(weighted_f1, 4))
print('Correct:', int((test_predictions == y_test).sum()))
print('Incorrect:', int((test_predictions != y_test).sum()))

print('\nClassification Report:')
print(classification_report(
    y_test,
    test_predictions,
    labels=['negatif', 'positif'],
    zero_division=0
))

print('Confusion Matrix:')
print(cm)

print('\n10 threshold terbaik berdasarkan validation:')
display(results_df.head(10))

print('\nLaporan tersimpan:', REPORT_DIR)

OPTIMASI THRESHOLD BERT
Threshold terbaik: 0.697
Validation accuracy: 87.31%

FINAL TEST DENGAN THRESHOLD TERPILIH
Test accuracy: 84.85%
Balanced accuracy: 82.13%
Macro F1: 0.8213
Weighted F1: 0.8485
Correct: 448
Incorrect: 80

Classification Report:
              precision    recall  f1-score   support

     negatif       0.89      0.89      0.89       367
     positif       0.75      0.75      0.75       161

    accuracy                           0.85       528
   macro avg       0.82      0.82      0.82       528
weighted avg       0.85      0.85      0.85       528

Confusion Matrix:
[[327  40]
 [ 40 121]]

10 threshold terbaik berdasarkan validation:


,threshold,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,0.697,0.873106,0.844819,0.848990,0.872306
1,0.698,0.873106,0.844819,0.848990,0.872306
2,0.699,0.873106,0.844819,0.848990,0.872306
3,0.700,0.873106,0.844819,0.848990,0.872306
4,0.499,0.871212,0.853775,0.850138,0.871851
5,0.500,0.871212,0.853775,0.850138,0.871851
6,0.501,0.871212,0.853775,0.850138,0.871851
7,0.502,0.871212,0.853775,0.850138,0.871851
8,0.503,0.871212,0.853775,0.850138,0.871851
9,0.504,0.871212,0.853775,0.850138,0.871851



Laporan tersimpan: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/reports/bert_base_uncased_cs2


## 5. Optimasi hyperparameter BERT

Learning rate dan konfigurasi training dibandingkan menggunakan validation set. Threshold dipilih dari validation, lalu diterapkan pada test.

In [11]:

import os
import json
import random
import shutil
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# =====================================================
# 1. KONFIGURASI
# =====================================================

SEED = 42
MODEL_NAME = "google-bert/bert-base-uncased"

LEARNING_RATES = [1e-5, 2e-5, 3e-5]
EPOCHS = 4
BATCH_SIZE = 16
MAX_LENGTH = 192
WEIGHT_DECAY = 0.01

LABEL2ID = {"negatif": 0, "positif": 1}
ID2LABEL = {0: "negatif", 1: "positif"}

PROJECT = Path(
    PROJECT
)

MODEL_DIR = PROJECT / "models"
REPORT_DIR = PROJECT / "reports/bert_hyperparameter_tuning"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_DIR = MODEL_DIR / "bert_tuned_best"

device = torch.device("cuda")
assert torch.cuda.is_available()

# =====================================================
# 2. LOAD DATA
# =====================================================

train_df = pd.read_csv(
    PROJECT / "data/final/steam_reviews_train_final.csv"
)

valid_df = pd.read_csv(
    PROJECT / "data/final/steam_reviews_validation_final.csv"
)

assert len(train_df) == 2462
assert len(valid_df) == 528

y_valid = valid_df["label"].map(LABEL2ID).values

print("Train:", len(train_df))
print("Validation:", len(valid_df))
print("GPU:", torch.cuda.get_device_name(0))

# =====================================================
# 3. TOKENIZER DAN DATASET
# =====================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):

    def __init__(self, df):
        self.encodings = tokenizer(
            df["text"].astype(str).tolist(),
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        self.labels = torch.tensor(
            df["label"].map(LABEL2ID).values,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: value[idx]
            for key, value in self.encodings.items()
        }
        item["labels"] = self.labels[idx]
        return item


train_dataset = ReviewDataset(train_df)
valid_dataset = ReviewDataset(valid_df)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

# =====================================================
# 4. FUNGSI EVALUASI VALIDATION
# =====================================================

THRESHOLDS = np.arange(0.45, 0.751, 0.01)

def evaluate_validation(model):

    model.eval()
    probabilities = []

    with torch.no_grad():

        for batch in valid_loader:

            batch = {
                k: v.to(device, non_blocking=True)
                for k, v in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**batch)

            probs = torch.softmax(
                outputs.logits.float(),
                dim=1
            )

            probabilities.extend(
                probs[:, LABEL2ID["positif"]].cpu().numpy()
            )

    probabilities = np.array(probabilities)

    best_result = None

    for threshold in THRESHOLDS:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        result = {
            "threshold": float(threshold),
            "validation_accuracy": accuracy_score(
                y_valid, predictions
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_valid, predictions
            ),
            "macro_f1": f1_score(
                y_valid, predictions, average="macro"
            ),
            "weighted_f1": f1_score(
                y_valid, predictions, average="weighted"
            )
        }

        if (
            best_result is None
            or (
                result["validation_accuracy"],
                result["macro_f1"]
            ) > (
                best_result["validation_accuracy"],
                best_result["macro_f1"]
            )
        ):
            best_result = result

    return best_result, probabilities

# =====================================================
# 5. TRAINING 3 LEARNING RATE
# =====================================================

all_results = []

best_overall = None
best_probabilities = None
best_model_config = None

for lr in LEARNING_RATES:

    print("\n" + "=" * 65)
    print(f"LEARNING RATE: {lr}")
    print("=" * 65)

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY
    )

    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(0.1 * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        return max(
            0.0,
            (total_steps - step) /
            max(1, total_steps - warmup_steps)
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda
    )

    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(EPOCHS):

        model.train()
        total_loss = 0

        for step, batch in enumerate(train_loader):

            batch = {
                k: v.to(device, non_blocking=True)
                for k, v in batch.items()
            }

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**batch)
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)

        # Validation dan threshold optimization
        metrics, probabilities = evaluate_validation(model)

        record = {
            "learning_rate": lr,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            **metrics
        }

        all_results.append(record)

        print(
            f"Epoch {epoch + 1}/{EPOCHS} "
            f"| Train Loss: {train_loss:.4f} "
            f"| Val Accuracy: "
            f"{metrics['validation_accuracy']*100:.2f}% "
            f"| Threshold: {metrics['threshold']:.2f} "
            f"| Macro F1: {metrics['macro_f1']:.4f}"
        )

        # Checkpoint terbaik berdasarkan validation
        score = (
            metrics["validation_accuracy"],
            metrics["macro_f1"]
        )

        current_best = (
            (-1, -1)
            if best_overall is None
            else (
                best_overall["validation_accuracy"],
                best_overall["macro_f1"]
            )
        )

        if score > current_best:

            best_overall = record.copy()
            best_probabilities = probabilities.copy()

            best_model_config = {
                "model": MODEL_NAME,
                "learning_rate": lr,
                "epoch": epoch + 1,
                "threshold": metrics["threshold"],
                "validation_accuracy": metrics["validation_accuracy"],
                "macro_f1": metrics["macro_f1"],
                "max_length": MAX_LENGTH
            }

            if BEST_MODEL_DIR.exists():
                shutil.rmtree(BEST_MODEL_DIR)

            model.save_pretrained(str(BEST_MODEL_DIR))
            tokenizer.save_pretrained(str(BEST_MODEL_DIR))

            print("CHECKPOINT GLOBAL TERBAIK DISIMPAN.")

    del model
    del optimizer
    del scheduler
    torch.cuda.empty_cache()

# =====================================================
# 6. SIMPAN HASIL
# =====================================================

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    ["validation_accuracy", "macro_f1"],
    ascending=False
)

results_df.to_csv(
    REPORT_DIR / "bert_hyperparameter_results.csv",
    index=False
)

with open(
    REPORT_DIR / "bert_best_hyperparameters.json",
    "w"
) as f:
    json.dump(best_model_config, f, indent=4)

# Simpan prediksi validation model terbaik
val_predictions = np.where(
    best_probabilities >= best_overall["threshold"],
    "positif",
    "negatif"
)

val_output = valid_df[
    ["review_id", "text", "label"]
].copy()

val_output["predicted_label"] = val_predictions
val_output["positive_probability"] = best_probabilities
val_output["threshold"] = best_overall["threshold"]

val_output.to_csv(
    REPORT_DIR / "bert_tuned_validation_predictions.csv",
    index=False
)

# =====================================================
# 7. HASIL
# =====================================================

print("\n" + "=" * 65)
print("BERT HYPERPARAMETER OPTIMIZATION SELESAI")
print("=" * 65)

print("\nTOP 5 MODEL:")
display(results_df.head(5))

print("\nKONFIGURASI TERBAIK:")
print(json.dumps(best_model_config, indent=4))

print("\nModel tersimpan:", BEST_MODEL_DIR)
print("Laporan tersimpan:", REPORT_DIR)

torch.cuda.empty_cache()


Train: 2462
Validation: 528
GPU: Tesla T4

LEARNING RATE: 1e-05


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4 | Train Loss: 0.5273 | Val Accuracy: 85.23% | Threshold: 0.55 | Macro F1: 0.8169


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CHECKPOINT GLOBAL TERBAIK DISIMPAN.
Epoch 2/4 | Train Loss: 0.3211 | Val Accuracy: 86.36% | Threshold: 0.68 | Macro F1: 0.8374


... (12 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CHECKPOINT GLOBAL TERBAIK DISIMPAN.
Epoch 3/4 | Train Loss: 0.1468 | Val Accuracy: 87.69% | Threshold: 0.49 | Macro F1: 0.8524
Epoch 4/4 | Train Loss: 0.0706 | Val Accuracy: 87.31% | Threshold: 0.62 | Macro F1: 0.8490

BERT HYPERPARAMETER OPTIMIZATION SELESAI

TOP 5 MODEL:


,learning_rate,epoch,train_loss,threshold,validation_accuracy,balanced_accuracy,macro_f1,weighted_f1
9,0.00003,2,0.275777,0.69,0.882576,0.858531,0.861005,0.882161
3,0.00001,4,0.176673,0.71,0.878788,0.840316,0.852184,0.876413
10,0.00003,3,0.146831,0.49,0.876894,0.845831,0.852425,0.875642
5,0.00002,2,0.296481,0.61,0.875000,0.847905,0.851511,0.874329
2,0.00001,3,0.229930,0.54,0.875000,0.841024,0.849312,0.873350



KONFIGURASI TERBAIK:
{
    "model": "google-bert/bert-base-uncased",
    "learning_rate": 3e-05,
    "epoch": 2,
    "threshold": 0.6900000000000002,
    "validation_accuracy": 0.8825757575757576,
    "macro_f1": 0.8610054347826087,
    "max_length": 192
}

Model tersimpan: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/models/bert_tuned_best
Laporan tersimpan: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/reports/bert_hyperparameter_tuning


In [12]:

import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# =====================================================
# 1. KONFIGURASI
# =====================================================

PROJECT = Path(
    PROJECT
)

MODEL_DIR = PROJECT / "models/bert_tuned_best"
REPORT_DIR = PROJECT / "reports/bert_hyperparameter_tuning"

TEST_PATH = PROJECT / "data/final/steam_reviews_test_final.csv"

THRESHOLD = 0.69
MAX_LENGTH = 192
BATCH_SIZE = 16

LABEL2ID = {"negatif": 0, "positif": 1}
ID2LABEL = {0: "negatif", 1: "positif"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert MODEL_DIR.exists(), f"Model tidak ditemukan: {MODEL_DIR}"
assert TEST_PATH.exists(), f"Dataset test tidak ditemukan: {TEST_PATH}"

# =====================================================
# 2. LOAD TEST DATA
# =====================================================

test_df = pd.read_csv(TEST_PATH)

assert len(test_df) == 528
assert test_df["review_id"].nunique() == 528
assert test_df["text"].notna().all()
assert test_df["label"].isin(LABEL2ID).all()

print("Test samples:", len(test_df))
print("Device:", device)
print("Threshold:", THRESHOLD)

# =====================================================
# 3. LOAD MODEL TERBAIK
# =====================================================

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))

model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR)
).to(device)

model.eval()

# =====================================================
# 4. TOKENIZATION
# =====================================================

class TestDataset(Dataset):

    def __init__(self, dataframe):
        self.encodings = tokenizer(
            dataframe["text"].astype(str).tolist(),
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        self.labels = torch.tensor(
            [LABEL2ID[x] for x in dataframe["label"]],
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {
            key: value[index]
            for key, value in self.encodings.items()
        }
        item["labels"] = self.labels[index]
        return item


test_dataset = TestDataset(test_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# =====================================================
# 5. PREDIKSI
# =====================================================

actual = []
positive_probabilities = []

with torch.no_grad():

    for batch in test_loader:

        labels = batch.pop("labels").numpy()

        batch = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=torch.cuda.is_available()
        ):
            outputs = model(**batch)

        probabilities = torch.softmax(
            outputs.logits.float(),
            dim=1
        )

        positive_index = LABEL2ID["positif"]

        positive_probabilities.extend(
            probabilities[:, positive_index].cpu().numpy()
        )

        actual.extend(labels)

actual = np.array(actual)
positive_probabilities = np.array(positive_probabilities)

# Gunakan threshold hasil validation
predictions = np.where(
    positive_probabilities >= THRESHOLD,
    LABEL2ID["positif"],
    LABEL2ID["negatif"]
)

y_true = [ID2LABEL[x] for x in actual]
y_pred = [ID2LABEL[x] for x in predictions]

# =====================================================
# 6. EVALUASI
# =====================================================

accuracy = accuracy_score(y_true, y_pred)
balanced = balanced_accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true, y_pred, average="macro", zero_division=0
)

recall = recall_score(
    y_true, y_pred, average="macro", zero_division=0
)

macro_f1 = f1_score(
    y_true, y_pred, average="macro", zero_division=0
)

weighted_f1 = f1_score(
    y_true, y_pred, average="weighted", zero_division=0
)

report = classification_report(
    y_true,
    y_pred,
    labels=["negatif", "positif"],
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["negatif", "positif"]
)

# =====================================================
# 7. SIMPAN PREDIKSI
# =====================================================

prediction_df = test_df[
    ["review_id", "review", "text", "label", "split"]
].copy()

prediction_df["predicted_label"] = y_pred
prediction_df["positive_probability"] = positive_probabilities
prediction_df["threshold"] = THRESHOLD
prediction_df["correct"] = (
    prediction_df["label"] == prediction_df["predicted_label"]
)

prediction_df.to_csv(
    REPORT_DIR / "bert_tuned_final_test_predictions.csv",
    index=False
)

# =====================================================
# 8. SIMPAN METRIK
# =====================================================

metrics = {
    "model": "BERT Base Uncased",
    "pretrained_model": "google-bert/bert-base-uncased",
    "learning_rate": 3e-5,
    "best_epoch": 2,
    "max_length": MAX_LENGTH,
    "threshold": THRESHOLD,
    "threshold_source": "validation set",
    "validation_accuracy": 0.882576,
    "test_samples": len(test_df),
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced),
    "precision_macro": float(precision),
    "recall_macro": float(recall),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "correct": int((predictions == actual).sum()),
    "incorrect": int((predictions != actual).sum()),
    "confusion_matrix": cm.tolist()
}

with open(
    REPORT_DIR / "bert_tuned_final_test_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=4)

pd.DataFrame(report).transpose().to_csv(
    REPORT_DIR / "bert_tuned_final_test_classification_report.csv"
)

pd.DataFrame(
    cm,
    index=["actual_negatif", "actual_positif"],
    columns=["predicted_negatif", "predicted_positif"]
).to_csv(
    REPORT_DIR / "bert_tuned_final_test_confusion_matrix.csv"
)

# =====================================================
# 9. TAMPILKAN HASIL
# =====================================================

print("\n" + "=" * 60)
print("BERT TUNED FINAL TEST EVALUATION")
print("=" * 60)

print("Learning rate: 3e-5")
print("Best epoch: 2")
print("Threshold:", THRESHOLD)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Balanced Accuracy: {balanced * 100:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=["negatif", "positif"],
    zero_division=0
))

print("Confusion Matrix:")
print(cm)

print("\nCorrect:", int((predictions == actual).sum()))
print("Incorrect:", int((predictions != actual).sum()))

print("\nModel:", MODEL_DIR)
print("Laporan:", REPORT_DIR)

print("\nEvaluasi selesai.")


Test samples: 528
Device: cuda
Threshold: 0.69


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


BERT TUNED FINAL TEST EVALUATION
Learning rate: 3e-5
Best epoch: 2
Threshold: 0.69
Accuracy: 84.66%
Balanced Accuracy: 81.47%
Macro F1: 0.8174
Weighted F1: 0.8459

Classification Report:
              precision    recall  f1-score   support

     negatif       0.88      0.90      0.89       367
     positif       0.76      0.73      0.74       161

    accuracy                           0.85       528
   macro avg       0.82      0.81      0.82       528
weighted avg       0.85      0.85      0.85       528

Confusion Matrix:
[[329  38]
 [ 43 118]]

Correct: 447
Incorrect: 81

Model: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/models/bert_tuned_best
Laporan: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/reports/bert_hyperparameter_tuning

Evaluasi selesai.


## 6. Training dan evaluasi RoBERTa

RoBERTa base di-fine-tune pada data training. Checkpoint terbaik dipilih berdasarkan validation.

In [13]:

import json
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# =====================================================
# 1. KONFIGURASI
# =====================================================

SEED = 42
MODEL_NAME = "FacebookAI/roberta-base"

EPOCHS = 3
BATCH_SIZE = 8
GRAD_ACCUMULATION = 2
MAX_LENGTH = 192
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

LABEL2ID = {"negatif": 0, "positif": 1}
ID2LABEL = {0: "negatif", 1: "positif"}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available()

device = torch.device("cuda")

PROJECT = Path(
    PROJECT
)

MODEL_DIR = PROJECT / "models/roberta_base_cs2"
REPORT_DIR = PROJECT / "reports/roberta_base_cs2"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# 2. LOAD DATASET
# =====================================================

train_df = pd.read_csv(
    PROJECT / "data/final/steam_reviews_train_final.csv"
)

valid_df = pd.read_csv(
    PROJECT / "data/final/steam_reviews_validation_final.csv"
)

assert len(train_df) == 2462
assert len(valid_df) == 528

y_valid = valid_df["label"].map(LABEL2ID).values

print("Train:", len(train_df))
print("Validation:", len(valid_df))
print("GPU:", torch.cuda.get_device_name(0))

# =====================================================
# 3. TOKENIZER
# =====================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# =====================================================
# 4. DATASET
# =====================================================

class ReviewDataset(Dataset):

    def __init__(self, dataframe):
        self.encodings = tokenizer(
            dataframe["text"].astype(str).tolist(),
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        self.labels = torch.tensor(
            dataframe["label"].map(LABEL2ID).values,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: value[idx]
            for key, value in self.encodings.items()
        }
        item["labels"] = self.labels[idx]
        return item


train_dataset = ReviewDataset(train_df)
valid_dataset = ReviewDataset(valid_df)

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# =====================================================
# 5. LOAD ROBERTA
# =====================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(device)

# =====================================================
# 6. OPTIMIZER DAN SCHEDULER
# =====================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

updates_per_epoch = (
    len(train_loader) + GRAD_ACCUMULATION - 1
) // GRAD_ACCUMULATION

total_steps = updates_per_epoch * EPOCHS
warmup_steps = int(0.1 * total_steps)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)

    return max(
        0.0,
        (total_steps - step) /
        max(1, total_steps - warmup_steps)
    )

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda
)

scaler = torch.amp.GradScaler("cuda")

# =====================================================
# 7. VALIDATION FUNCTION
# =====================================================

THRESHOLDS = np.round(
    np.arange(0.40, 0.701, 0.01),
    3
)

def evaluate_validation(model):

    model.eval()
    probabilities = []
    actual = []

    with torch.no_grad():

        for batch in valid_loader:

            labels = batch.pop("labels").numpy()

            batch = {
                key: value.to(device, non_blocking=True)
                for key, value in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**batch)

            probs = torch.softmax(
                outputs.logits.float(),
                dim=1
            )

            probabilities.extend(
                probs[:, LABEL2ID["positif"]].cpu().numpy()
            )

            actual.extend(labels)

    probabilities = np.array(probabilities)
    actual = np.array(actual)

    threshold_results = []

    for threshold in THRESHOLDS:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        threshold_results.append({
            "threshold": float(threshold),
            "accuracy": accuracy_score(
                actual, predictions
            ),
            "balanced_accuracy": balanced_accuracy_score(
                actual, predictions
            ),
            "macro_f1": f1_score(
                actual, predictions, average="macro"
            ),
            "weighted_f1": f1_score(
                actual, predictions, average="weighted"
            )
        })

    threshold_df = pd.DataFrame(threshold_results)

    threshold_df = threshold_df.sort_values(
        ["accuracy", "macro_f1"],
        ascending=False
    ).reset_index(drop=True)

    return threshold_df.iloc[0].to_dict(), probabilities, threshold_df

# =====================================================
# 8. TRAINING
# =====================================================

history = []

best_score = (-1, -1)
best_config = None
best_probabilities = None

for epoch in range(EPOCHS):

    print("\n" + "=" * 65)
    print(f"ROBERTA EPOCH {epoch + 1}/{EPOCHS}")
    print("=" * 65)

    model.train()
    total_train_loss = 0

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader):

        batch = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(
            loss / GRAD_ACCUMULATION
        ).backward()

        total_train_loss += loss.item()

        should_update = (
            (step + 1) % GRAD_ACCUMULATION == 0
            or (step + 1) == len(train_loader)
        )

        if should_update:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            optimizer.zero_grad(set_to_none=True)

        if (step + 1) % 50 == 0:
            print(
                f"Batch {step + 1}/{len(train_loader)} "
                f"| Loss: {loss.item():.4f}"
            )

    train_loss = total_train_loss / len(train_loader)

    # VALIDATION
    metrics, probabilities, threshold_df = evaluate_validation(model)

    record = {
        "epoch": epoch + 1,
        "learning_rate": LEARNING_RATE,
        "train_loss": train_loss,
        **metrics
    }

    history.append(record)

    print(f"\nTrain Loss: {train_loss:.4f}")
    print(f"Threshold: {metrics['threshold']:.3f}")
    print(
        f"Validation Accuracy: "
        f"{metrics['accuracy']*100:.2f}%"
    )
    print(
        f"Balanced Accuracy: "
        f"{metrics['balanced_accuracy']:.4f}"
    )
    print(f"Macro F1: {metrics['macro_f1']:.4f}")
    print(f"Weighted F1: {metrics['weighted_f1']:.4f}")

    # CHECKPOINT TERBAIK
    current_score = (
        metrics["accuracy"],
        metrics["macro_f1"]
    )

    if current_score > best_score:

        best_score = current_score
        best_config = record.copy()
        best_probabilities = probabilities.copy()

        model.save_pretrained(str(MODEL_DIR))
        tokenizer.save_pretrained(str(MODEL_DIR))

        threshold_df.to_csv(
            REPORT_DIR / "roberta_threshold_search_validation.csv",
            index=False
        )

        print("CHECKPOINT TERBAIK DISIMPAN.")

    pd.DataFrame(history).to_csv(
        REPORT_DIR / "roberta_training_history.csv",
        index=False
    )

# =====================================================
# 9. SIMPAN PREDIKSI VALIDATION TERBAIK
# =====================================================

best_threshold = best_config["threshold"]

val_predictions = np.where(
    best_probabilities >= best_threshold,
    "positif",
    "negatif"
)

val_output = valid_df[
    ["review_id", "text", "label"]
].copy()

val_output["predicted_label"] = val_predictions
val_output["positive_probability"] = best_probabilities
val_output["threshold"] = best_threshold

val_output.to_csv(
    REPORT_DIR / "roberta_validation_predictions.csv",
    index=False
)

with open(
    REPORT_DIR / "roberta_best_config.json",
    "w"
) as f:
    json.dump(best_config, f, indent=4)

# =====================================================
# 10. HASIL
# =====================================================

print("\n" + "=" * 65)
print("ROBERTA FINE-TUNING SELESAI")
print("=" * 65)

print("Best Epoch:", best_config["epoch"])
print("Learning Rate:", LEARNING_RATE)
print("Best Threshold:", best_threshold)

print(
    "Best Validation Accuracy:",
    f"{best_config['accuracy']*100:.2f}%"
)

print("Best Macro F1:", round(best_config["macro_f1"], 4))

print("\nTraining History:")
display(pd.DataFrame(history))

print("\nModel:", MODEL_DIR)
print("Reports:", REPORT_DIR)

torch.cuda.empty_cache()


Train: 2462
Validation: 528
GPU: Tesla T4


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (9 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CHECKPOINT TERBAIK DISIMPAN.

ROBERTA FINE-TUNING SELESAI
Best Epoch: 3
Learning Rate: 2e-05
Best Threshold: 0.5
Best Validation Accuracy: 88.26%
Best Macro F1: 0.861

Training History:


,epoch,learning_rate,train_loss,threshold,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,1,0.00002,0.499142,0.43,0.856061,0.803279,0.820125,0.851188
1,2,0.00002,0.327891,0.66,0.869318,0.818002,0.836345,0.864727
2,3,0.00002,0.242828,0.50,0.882576,0.858531,0.861005,0.882161



Model: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/models/roberta_base_cs2
Reports: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/reports/roberta_base_cs2


In [14]:

import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# =====================================================
# 1. PATH DAN KONFIGURASI
# =====================================================

PROJECT = Path(
    PROJECT
)

MODEL_DIR = PROJECT / "models/roberta_base_cs2"
REPORT_DIR = PROJECT / "reports/roberta_base_cs2"
TEST_PATH = PROJECT / "data/final/steam_reviews_test_final.csv"

THRESHOLD = 0.50
MAX_LENGTH = 192
BATCH_SIZE = 8

LABEL2ID = {"negatif": 0, "positif": 1}
ID2LABEL = {0: "negatif", 1: "positif"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert MODEL_DIR.exists(), f"Model tidak ditemukan: {MODEL_DIR}"
assert TEST_PATH.exists(), f"Dataset tidak ditemukan: {TEST_PATH}"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# 2. LOAD TEST DATA
# =====================================================

test_df = pd.read_csv(TEST_PATH)

assert len(test_df) == 528
assert test_df["review_id"].nunique() == 528
assert test_df["text"].notna().all()
assert test_df["label"].isin(LABEL2ID).all()

print("Test samples:", len(test_df))
print("Device:", device)
print("Threshold:", THRESHOLD)

# =====================================================
# 3. LOAD MODEL DAN TOKENIZER
# =====================================================

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))

model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR)
).to(device)

model.eval()

# =====================================================
# 4. TEST DATASET
# =====================================================

class TestDataset(Dataset):

    def __init__(self, dataframe):
        self.encodings = tokenizer(
            dataframe["text"].astype(str).tolist(),
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        self.labels = torch.tensor(
            dataframe["label"].map(LABEL2ID).values,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {
            key: value[index]
            for key, value in self.encodings.items()
        }

        item["labels"] = self.labels[index]
        return item


test_dataset = TestDataset(test_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# =====================================================
# 5. PREDIKSI
# =====================================================

actual = []
positive_probabilities = []

with torch.no_grad():

    for batch in test_loader:

        labels = batch.pop("labels").numpy()

        batch = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=torch.cuda.is_available()
        ):
            outputs = model(**batch)

        probabilities = torch.softmax(
            outputs.logits.float(),
            dim=1
        )

        positive_probabilities.extend(
            probabilities[:, LABEL2ID["positif"]].cpu().numpy()
        )

        actual.extend(labels)

actual = np.array(actual)
positive_probabilities = np.array(positive_probabilities)

# Threshold berasal dari validation set
predictions = np.where(
    positive_probabilities >= THRESHOLD,
    LABEL2ID["positif"],
    LABEL2ID["negatif"]
)

y_true = [ID2LABEL[x] for x in actual]
y_pred = [ID2LABEL[x] for x in predictions]

# =====================================================
# 6. METRIK EVALUASI
# =====================================================

accuracy = accuracy_score(y_true, y_pred)

balanced = balanced_accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true, y_pred,
    average="weighted",
    zero_division=0
)

report = classification_report(
    y_true,
    y_pred,
    labels=["negatif", "positif"],
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["negatif", "positif"]
)

# =====================================================
# 7. SIMPAN PREDIKSI
# =====================================================

prediction_df = test_df[
    ["review_id", "review", "text", "label", "split"]
].copy()

prediction_df["predicted_label"] = y_pred
prediction_df["positive_probability"] = positive_probabilities
prediction_df["threshold"] = THRESHOLD
prediction_df["correct"] = (
    prediction_df["label"] == prediction_df["predicted_label"]
)

prediction_df.to_csv(
    REPORT_DIR / "roberta_final_test_predictions.csv",
    index=False
)

# =====================================================
# 8. SIMPAN METRIK
# =====================================================

metrics = {
    "model": "RoBERTa Base",
    "pretrained_model": "FacebookAI/roberta-base",
    "learning_rate": 2e-5,
    "best_epoch": 3,
    "max_length": MAX_LENGTH,
    "threshold": THRESHOLD,
    "threshold_source": "validation set",
    "validation_accuracy": 0.882576,
    "test_samples": len(test_df),
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced),
    "precision_macro": float(precision),
    "recall_macro": float(recall),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "correct": int((predictions == actual).sum()),
    "incorrect": int((predictions != actual).sum()),
    "confusion_matrix": cm.tolist()
}

with open(
    REPORT_DIR / "roberta_final_test_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=4)

pd.DataFrame(report).transpose().to_csv(
    REPORT_DIR / "roberta_final_test_classification_report.csv"
)

pd.DataFrame(
    cm,
    index=["actual_negatif", "actual_positif"],
    columns=["predicted_negatif", "predicted_positif"]
).to_csv(
    REPORT_DIR / "roberta_final_test_confusion_matrix.csv"
)

# =====================================================
# 9. TAMPILKAN HASIL
# =====================================================

print("\n" + "=" * 60)
print("ROBERTA FINAL TEST EVALUATION")
print("=" * 60)

print("Learning rate: 2e-5")
print("Best epoch: 3")
print("Threshold:", THRESHOLD)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Balanced Accuracy: {balanced * 100:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=["negatif", "positif"],
    zero_division=0
))

print("Confusion Matrix:")
print(cm)

print("\nCorrect:", int((predictions == actual).sum()))
print("Incorrect:", int((predictions != actual).sum()))

print("\nModel:", MODEL_DIR)
print("Reports:", REPORT_DIR)
print("\nEvaluasi selesai.")


Test samples: 528
Device: cuda
Threshold: 0.5


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


ROBERTA FINAL TEST EVALUATION
Learning rate: 2e-5
Best epoch: 3
Threshold: 0.5
Accuracy: 84.28%
Balanced Accuracy: 81.37%
Macro F1: 0.8143
Weighted F1: 0.8427

Classification Report:
              precision    recall  f1-score   support

     negatif       0.89      0.89      0.89       367
     positif       0.74      0.74      0.74       161

    accuracy                           0.84       528
   macro avg       0.81      0.81      0.81       528
weighted avg       0.84      0.84      0.84       528

Confusion Matrix:
[[326  41]
 [ 42 119]]

Correct: 445
Incorrect: 83

Model: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/models/roberta_base_cs2
Reports: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/reports/roberta_base_cs2

Evaluasi selesai.


## 7. Ensemble BERT dan RoBERTa

Bobot kedua model dan threshold ditentukan menggunakan validation set. Konfigurasi yang dipilih kemudian dievaluasi pada test set.

In [15]:

import json
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# =====================================================
# 1. PATH
# =====================================================

PROJECT = Path(
    PROJECT
)

BERT_DIR = PROJECT / 'reports/bert_hyperparameter_tuning'
ROBERTA_DIR = PROJECT / 'reports/roberta_base_cs2'

OUTPUT_DIR = PROJECT / 'reports/ensemble_bert_roberta'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# 2. LOAD PREDICTIONS
# =====================================================

bert_val = pd.read_csv(
    BERT_DIR / 'bert_tuned_validation_predictions.csv'
)

roberta_val = pd.read_csv(
    ROBERTA_DIR / 'roberta_validation_predictions.csv'
)

bert_test = pd.read_csv(
    BERT_DIR / 'bert_tuned_final_test_predictions.csv'
)

roberta_test = pd.read_csv(
    ROBERTA_DIR / 'roberta_final_test_predictions.csv'
)

def prepare(df, model_name):
    return df[
        ['review_id', 'label', 'positive_probability']
    ].rename(columns={
        'label': f'label_{model_name}',
        'positive_probability': f'prob_{model_name}'
    })

# Align validation predictions by review ID
val = prepare(bert_val, 'bert').merge(
    prepare(roberta_val, 'roberta'),
    on='review_id',
    how='inner'
)

# Align test predictions by review ID
test = prepare(bert_test, 'bert').merge(
    prepare(roberta_test, 'roberta'),
    on='review_id',
    how='inner'
)

assert len(val) == 528
assert len(test) == 528
assert val['review_id'].nunique() == 528
assert test['review_id'].nunique() == 528

assert (
    val['label_bert'] == val['label_roberta']
).all()

assert (
    test['label_bert'] == test['label_roberta']
).all()

y_val = val['label_bert'].values
y_test = test['label_bert'].values

print("Validation samples:", len(val))
print("Test samples:", len(test))
print("ID alignment: OK")

# =====================================================
# 3. ENSEMBLE SEARCH PADA VALIDATION
# =====================================================

weights = [0.0, 0.25, 0.50, 0.75, 1.0]
thresholds = np.round(
    np.arange(0.35, 0.751, 0.005), 3
)

results = []

for bert_weight in weights:

    ensemble_prob = (
        bert_weight * val['prob_bert'].values
        + (1 - bert_weight) * val['prob_roberta'].values
    )

    for threshold in thresholds:

        pred = np.where(
            ensemble_prob >= threshold,
            'positif',
            'negatif'
        )

        results.append({
            'bert_weight': bert_weight,
            'roberta_weight': 1 - bert_weight,
            'threshold': float(threshold),
            'validation_accuracy': accuracy_score(y_val, pred),
            'balanced_accuracy': balanced_accuracy_score(y_val, pred),
            'macro_f1': f1_score(
                y_val, pred, average='macro'
            ),
            'weighted_f1': f1_score(
                y_val, pred, average='weighted'
            )
        })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    ['validation_accuracy', 'macro_f1', 'balanced_accuracy'],
    ascending=False
).reset_index(drop=True)

# Pilih kombinasi terbaik hanya berdasarkan validation
best = results_df.iloc[0]

BEST_BERT_WEIGHT = float(best['bert_weight'])
BEST_ROBERTA_WEIGHT = float(best['roberta_weight'])
BEST_THRESHOLD = float(best['threshold'])

results_df.to_csv(
    OUTPUT_DIR / 'ensemble_validation_search.csv',
    index=False
)

best_config = {
    'model': 'BERT + RoBERTa Ensemble',
    'bert_weight': BEST_BERT_WEIGHT,
    'roberta_weight': BEST_ROBERTA_WEIGHT,
    'threshold': BEST_THRESHOLD,
    'validation_accuracy': float(best['validation_accuracy']),
    'validation_macro_f1': float(best['macro_f1'])
}

with open(OUTPUT_DIR / 'ensemble_best_config.json', 'w') as f:
    json.dump(best_config, f, indent=4)

print("\n=== TOP 10 ENSEMBLE VALIDATION ===")
display(results_df.head(10))

print("\n=== BEST CONFIGURATION ===")
print(json.dumps(best_config, indent=4))

# =====================================================
# 4. FINAL TEST EVALUATION
# =====================================================

test_prob = (
    BEST_BERT_WEIGHT * test['prob_bert'].values
    + BEST_ROBERTA_WEIGHT * test['prob_roberta'].values
)

test_predictions = np.where(
    test_prob >= BEST_THRESHOLD,
    'positif',
    'negatif'
)

accuracy = accuracy_score(y_test, test_predictions)

balanced = balanced_accuracy_score(
    y_test, test_predictions
)

macro_f1 = f1_score(
    y_test, test_predictions, average='macro'
)

weighted_f1 = f1_score(
    y_test, test_predictions, average='weighted'
)

cm = confusion_matrix(
    y_test,
    test_predictions,
    labels=['negatif', 'positif']
)

# Simpan prediksi final
test_output = test[
    ['review_id', 'label_bert']
].rename(columns={'label_bert': 'actual_label'}).copy()

test_output['predicted_label'] = test_predictions
test_output['positive_probability'] = test_prob
test_output['threshold'] = BEST_THRESHOLD
test_output['correct'] = (
    test_output['actual_label'] == test_output['predicted_label']
)

test_output.to_csv(
    OUTPUT_DIR / 'ensemble_test_predictions.csv',
    index=False
)

test_metrics = {
    **best_config,
    'test_samples': len(test),
    'test_accuracy': float(accuracy),
    'test_balanced_accuracy': float(balanced),
    'test_macro_f1': float(macro_f1),
    'test_weighted_f1': float(weighted_f1),
    'correct': int((test_predictions == y_test).sum()),
    'incorrect': int((test_predictions != y_test).sum()),
    'confusion_matrix': cm.tolist()
}

with open(OUTPUT_DIR / 'ensemble_test_metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=4)

# =====================================================
# 5. TAMPILKAN HASIL
# =====================================================

print("\n" + "=" * 60)
print("ENSEMBLE FINAL TEST EVALUATION")
print("=" * 60)

print("BERT weight:", BEST_BERT_WEIGHT)
print("RoBERTa weight:", BEST_ROBERTA_WEIGHT)
print("Threshold:", BEST_THRESHOLD)

print(f"\nTest Accuracy: {accuracy * 100:.2f}%")
print(f"Balanced Accuracy: {balanced * 100:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    test_predictions,
    labels=['negatif', 'positif'],
    zero_division=0
))

print("Confusion Matrix:")
print(cm)

print("\nCorrect:", int((test_predictions == y_test).sum()))
print("Incorrect:", int((test_predictions != y_test).sum()))

print("\nLaporan tersimpan:", OUTPUT_DIR)


Validation samples: 528
Test samples: 528
ID alignment: OK

=== TOP 10 ENSEMBLE VALIDATION ===


,bert_weight,roberta_weight,threshold,validation_accuracy,balanced_accuracy,macro_f1,weighted_f1
0,0.50,0.50,0.585,0.892045,0.861921,0.870105,0.890731
1,0.50,0.50,0.590,0.892045,0.861921,0.870105,0.890731
2,0.50,0.50,0.595,0.890152,0.858834,0.867578,0.888702
3,0.50,0.50,0.615,0.890152,0.855394,0.866562,0.888239
4,0.50,0.50,0.620,0.890152,0.855394,0.866562,0.888239
5,0.50,0.50,0.625,0.890152,0.855394,0.866562,0.888239
6,0.75,0.25,0.505,0.888258,0.871231,0.869320,0.888540
7,0.50,0.50,0.580,0.888258,0.859188,0.866047,0.887121
8,0.50,0.50,0.600,0.888258,0.855748,0.865039,0.886667
9,0.50,0.50,0.605,0.888258,0.854028,0.864523,0.886432



=== BEST CONFIGURATION ===
{
    "model": "BERT + RoBERTa Ensemble",
    "bert_weight": 0.5,
    "roberta_weight": 0.5,
    "threshold": 0.585,
    "validation_accuracy": 0.8920454545454546,
    "validation_macro_f1": 0.8701050950603164
}

ENSEMBLE FINAL TEST EVALUATION
BERT weight: 0.5
RoBERTa weight: 0.5
Threshold: 0.585

Test Accuracy: 85.04%
Balanced Accuracy: 81.39%
Macro F1: 0.8200
Weighted F1: 0.8488

Classification Report:
              precision    recall  f1-score   support

     negatif       0.88      0.91      0.89       367
     positif       0.77      0.72      0.75       161

    accuracy                           0.85       528
   macro avg       0.83      0.81      0.82       528
weighted avg       0.85      0.85      0.85       528

Confusion Matrix:
[[333  34]
 [ 45 116]]

Correct: 449
Incorrect: 79

Laporan tersimpan: /content/gdrive_verified_e1cccd02/MyDrive/Proyek_Analisis_Sentimen/reports/ensemble_bert_roberta


## 8. Pengujian inference

In [16]:
import sys
import torch
from pathlib import Path

sys.path.insert(0, str(PROJECT / 'src'))
from inference import CS2ReviewClassifier

print('Device:', 'GPU' if torch.cuda.is_available() else 'CPU')
classifier = CS2ReviewClassifier(PROJECT)

samples = [
    'This game is amazing and very fun to play!',
    'Too many cheaters and constant crashes. Terrible experience.',
    'The gameplay is fun, but matchmaking is frustrating.'
]

for review in samples:
    result = classifier.predict(review)
    print('\nUlasan:', result['review'])
    print('Prediksi:', result['predicted_label'])
    print('Probabilitas positif:', result['positive_probability'])
    print('Probabilitas negatif:', result['negative_probability'])


Device: CPU

Ulasan: This game is amazing and very fun to play!
Prediksi: positif
Probabilitas positif: 0.9891
Probabilitas negatif: 0.0109

Ulasan: Too many cheaters and constant crashes. Terrible experience.
Prediksi: negatif
Probabilitas positif: 0.0196
Probabilitas negatif: 0.9804

Ulasan: The gameplay is fun, but matchmaking is frustrating.
Prediksi: positif
Probabilitas positif: 0.9084
Probabilitas negatif: 0.0916


## 9. Ringkasan, interpretasi, dan keterbatasan

Model akhir adalah ensemble BERT fine-tuned dan RoBERTa base dengan bobot 50:50 dan threshold 0,585. Pada validation set, akurasi ensemble adalah 89,20%; pada test set, akurasi yang teramati adalah **85,04% (449/528)**, macro F1 0,8200, dan weighted F1 0,8488.

**Batasan metodologis:** (1) Target model adalah keputusan rekomendasi Steam (`voted_up`), bukan anotasi polaritas teks manusia. (2) Anotasi AI dipakai sebagai bantuan audit bahasa/kualitas, bukan ground truth target. (3) Test set pernah digunakan dalam rangkaian perbandingan kandidat model, sehingga angka test ini bukan estimasi holdout yang sepenuhnya independen. (4) Dataset mentah berjumlah 10.000, tetapi dataset final yang digunakan untuk training berjumlah 3.518 ulasan dan hanya memiliki dua kelas.

Checkpoint BERT dan RoBERTa tidak disertakan di dalam ZIP karena ukuran model besar. Notebook berisi proses training untuk menghasilkan checkpoint tersebut; `src/inference.py` dapat dijalankan setelah checkpoint dan konfigurasi ensemble tersedia.
